In [ ]:
! pip install xarray rioxarray netCDF4 dask

In [ ]:
import os
import gzip
import tempfile
import re
import xarray as xr
import rioxarray
import pandas as pd
from pathlib import Path

def aggregate_gosif_monthly(input_dir, output_nc_path):
    """
    Aggregates compressed monthly GOSIF GeoTIFFs into a single NetCDF file.
    """
    input_dir = Path(input_dir)

    # Regex to extract year and month from filename: GOSIF_GPP_2000.M03_Mean.tif.gz
    file_pattern = re.compile(r'GOSIF_GPP_(\d{4})\.M(\d{2})_Mean\.tif\.gz')

    # Find all matching files
    files = sorted(list(input_dir.glob('GOSIF_GPP_*.tif.gz')))

    if not files:
        print(f"No GOSIF files found in {input_dir}")
        return

    print(f"Found {len(files)} monthly files. Starting aggregation...")

    datasets = []

    for f in files:
        match = file_pattern.search(f.name)
        if not match:
            print(f"  Skipping {f.name} (filename pattern mismatch)")
            continue

        year, month = int(match.group(1)), int(match.group(2))
        # Create a pandas timestamp for the 1st day of that month
        time_coord = pd.Timestamp(year=year, month=month, day=1)

        print(f"  Processing: {f.name} -> {time_coord.strftime('%Y-%m')}")

        # 1. Unzip the .gz file to a temporary file on disk
        # (Doing this on disk is more memory-efficient than RAM for large rasters)
        with gzip.open(f, 'rb') as gz_file:
            with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp_tif:
                tmp_tif.write(gz_file.read())
                tmp_path = tmp_tif.name

        try:
            # 2. Read the TIFF using rioxarray
            # masked=True applies the NoData value as NaN
            da = rioxarray.open_rasterio(tmp_path, masked=True)

            # Remove the 'band' dimension (since it's a single-band mean file)
            if 'band' in da.dims:
                da = da.squeeze('band', drop=True)

            # Rename the data variable
            da.name = 'GOSIF_GPP'

            # 3. Assign the time coordinate and expand dimensions
            da = da.expand_dims(time=[time_coord])

            datasets.append(da)

        finally:
            # 4. Clean up the temporary file immediately to save disk space
            if os.path.exists(tmp_path):
                os.remove(tmp_path)

    if not datasets:
        print("No valid datasets were processed. Exiting.")
        return

    print("\nConcatenating datasets along the time dimension...")
    # Concatenate all DataArrays along the new time dimension
    combined_ds = xr.concat(datasets, dim='time')

    # Sort by time just in case files were processed out of order
    combined_ds = combined_ds.sortby('time')

    # Add metadata (CF-Compliantish)
    combined_ds.attrs = {
        'title': 'GOSIF Gross Primary Production (Monthly Mean)',
        'source': 'Aggregated from individual GOSIF monthly .tif.gz files',
        'history': f'Created on {pd.Timestamp.now().isoformat()}',
        'units': 'g C m-2 d-1'  # Adjust if your GPP units differ
    }

    print(f"Saving aggregated dataset to: {output_nc_path}")
    # Use NetCDF4 format with compression to save space
    encoding = {
        'GOSIF_GPP': {
            'zlib': True,
            'complevel': 4,
            'shuffle': True,
            '_FillValue': -9999.0
        }
    }

    # Ensure output directory exists
    Path(output_nc_path).parent.mkdir(parents=True, exist_ok=True)

    combined_ds.to_netcdf(
        output_nc_path,
        format='NETCDF4',
        encoding=encoding
    )

    print("Aggregation complete!")

# ─── EXECUTE ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # UPDATE THESE PATHS
    input_folder = r"./path/to/your/gosif/files"
    output_file = r"./path/to/output/GOSIF_GPP_Monthly_Aggregated.nc"

    aggregate_gosif_monthly(input_folder, output_file)